In [1]:
# Uvoz biblioteka

import psycopg2
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium import plugins
import rasterio
from rasterio.plot import show
from shapely.geometry import Point, Polygon, LineString, box
from shapely.ops import unary_union
import os
import warnings
warnings.filterwarnings('ignore')

print(" Sve biblioteke uspešno uvezene!")

 Sve biblioteke uspešno uvezene!


In [2]:
# Povezivanje na bazu podataka

DB_CONFIG = {
    'dbname': 'sumski_pozari_db',
    'user': 'postgres',
    'password': 'milica123',  
    'host': 'localhost',
    'port': '5432'
}


conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()
print(" Povezivanje na bazu uspešno!")

# Provera PostGIS-a
cur.execute("SELECT postgis_version();")
version = cur.fetchone()
print(f" PostGIS verzija: {version[0]}")

 Povezivanje na bazu uspešno!
 PostGIS verzija: 3.6 USE_GEOS=1 USE_PROJ=1 USE_STATS=1


In [3]:
# Kreiranje tabela u bazi podataka

tables = ['opozarena_podrucja', 'pozari', 'meteo_merenja', 
          'meteo_stanice', 'vatrogasne_jedinice', 'sumske_parcele', 'opstine']

for table in tables:
    cur.execute(f"DROP TABLE IF EXISTS {table} CASCADE;")
conn.commit()
print(" Stare tabele obrisane")

# Opštine
cur.execute("""
CREATE TABLE opstine (
    id SERIAL PRIMARY KEY,
    naziv VARCHAR(100) NOT NULL,
    region VARCHAR(100),
    povrsina_km2 NUMERIC(10,2),
    broj_stanovnika INTEGER,
    geom GEOMETRY(MultiPolygon, 4326)
);
""")

# Šumske parcele
cur.execute("""
CREATE TABLE sumske_parcele (
    id SERIAL PRIMARY KEY,
    naziv VARCHAR(100) NOT NULL,
    tip_sume VARCHAR(50),
    povrsina_ha NUMERIC(10,2),
    gustina_sume VARCHAR(20),
    nadmorska_visina INTEGER,
    opstina_id INTEGER REFERENCES opstine(id),
    geom GEOMETRY(Polygon, 4326)
);
""")

# Vatrogasne jedinice
cur.execute("""
CREATE TABLE vatrogasne_jedinice (
    id SERIAL PRIMARY KEY,
    naziv VARCHAR(100) NOT NULL,
    adresa TEXT,
    broj_vozila INTEGER,
    broj_osoblja INTEGER,
    radno_vreme VARCHAR(20),
    opstina_id INTEGER REFERENCES opstine(id),
    geom GEOMETRY(Point, 4326)
);
""")

# Meteorološke stanice
cur.execute("""
CREATE TABLE meteo_stanice (
    id SERIAL PRIMARY KEY,
    naziv VARCHAR(100) NOT NULL,
    visina_m INTEGER,
    tip_stanice VARCHAR(20),
    opstina_id INTEGER REFERENCES opstine(id),
    geom GEOMETRY(Point, 4326)
);
""")

# Meteorološka merenja
cur.execute("""
CREATE TABLE meteo_merenja (
    id SERIAL PRIMARY KEY,
    stanica_id INTEGER REFERENCES meteo_stanice(id),
    datum DATE NOT NULL,
    temperatura NUMERIC(5,2),
    vlaznost_vazduha NUMERIC(5,2),
    brzina_vetra NUMERIC(5,2),
    padavine_mm NUMERIC(5,2)
);
""")

# Požari
cur.execute("""
CREATE TABLE pozari (
    id SERIAL PRIMARY KEY,
    datum_pocetka DATE NOT NULL,
    datum_gasenja DATE,
    parcela_id INTEGER REFERENCES sumske_parcele(id),
    uzrok TEXT,
    intenzitet VARCHAR(20),
    povrsina_zahvacena_ha NUMERIC(10,2),
    broj_angazovanih_vatrogasaca INTEGER
);
""")

# Opožarena područja
cur.execute("""
CREATE TABLE opozarena_podrucja (
    id SERIAL PRIMARY KEY,
    pozar_id INTEGER REFERENCES pozari(id),
    datum_detekcije DATE,
    metoda_detekcije VARCHAR(50),
    povrsina_ha NUMERIC(10,2),
    indeks_dnbr NUMERIC(5,4),
    geom GEOMETRY(Polygon, 4326)
);
""")

conn.commit()
print(" Sve tabele uspešno kreirane!")

 Stare tabele obrisane
 Sve tabele uspešno kreirane!


In [4]:
# Unos podataka u tabelu opštine

opstine = [
    ("Pirot", "Stara planina", 1232.00, 56700,
     "MULTIPOLYGON(((22.45 43.05, 22.75 43.05, 22.75 43.25, 22.45 43.25, 22.45 43.05)))"),
    ("Babušnica", "Stara planina", 524.00, 9800,
     "MULTIPOLYGON(((22.30 43.02, 22.55 43.02, 22.55 43.22, 22.30 43.22, 22.30 43.02)))"),
    ("Dimitrovgrad", "Stara planina", 480.00, 8300,
     "MULTIPOLYGON(((22.62 42.92, 22.88 42.92, 22.88 43.12, 22.62 43.12, 22.62 42.92)))"),
    ("Knjaževac", "Stara planina", 1202.00, 28400,
     "MULTIPOLYGON(((22.10 43.42, 22.45 43.42, 22.45 43.67, 22.10 43.67, 22.10 43.42)))"),
    ("Zaječar", "Timočka regija", 1059.00, 50700,
     "MULTIPOLYGON(((22.13 43.78, 22.45 43.78, 22.45 44.00, 22.13 44.00, 22.13 43.78)))"),
]

for naziv, region, povrsina, stanovnika, geom in opstine:
    cur.execute("""
        INSERT INTO opstine (naziv, region, povrsina_km2, broj_stanovnika, geom)
        VALUES (%s, %s, %s, %s, ST_GeomFromText(%s, 4326))
    """, (naziv, region, povrsina, stanovnika, geom))

conn.commit()
print("Uneto 5 opština")

Uneto 5 opština


In [5]:
# Unos podataka u tabelu šumske parcele

parcele = [
    ("Parcela Vlasina", "bukova šuma", 340.50, "gusta", 1100, 1,
     "POLYGON((22.55 43.10, 22.60 43.10, 22.60 43.15, 22.55 43.15, 22.55 43.10))"),
    ("Parcela Rsovci", "mesovita šuma", 210.20, "srednja", 850, 1,
     "POLYGON((22.48 43.07, 22.52 43.07, 22.52 43.11, 22.48 43.11, 22.48 43.07))"),
    ("Parcela Petrlaš", "borova šuma", 150.00, "retka", 950, 2,
     "POLYGON((22.35 43.05, 22.40 43.05, 22.40 43.10, 22.35 43.10, 22.35 43.05))"),
    ("Parcela Strelac", "bukova šuma", 275.80, "gusta", 1200, 3,
     "POLYGON((22.70 42.97, 22.75 42.97, 22.75 43.02, 22.70 43.02, 22.70 42.97))"),
    ("Parcela Stara planina sever", "mesovita šuma", 410.00, "gusta", 1350, 4,
     "POLYGON((22.20 43.50, 22.28 43.50, 22.28 43.58, 22.20 43.58, 22.20 43.50))"),
    ("Parcela Crni vrh", "borova šuma", 198.40, "srednja", 1050, 5,
     "POLYGON((22.20 43.85, 22.27 43.85, 22.27 43.92, 22.20 43.92, 22.20 43.85))"),
]

for naziv, tip, povrsina, gustina, nadmorska, opstina_id, geom in parcele:
    cur.execute("""
        INSERT INTO sumske_parcele 
        (naziv, tip_sume, povrsina_ha, gustina_sume, nadmorska_visina, opstina_id, geom)
        VALUES (%s, %s, %s, %s, %s, %s, ST_GeomFromText(%s, 4326))
    """, (naziv, tip, povrsina, gustina, nadmorska, opstina_id, geom))

conn.commit()
print(" Uneto 6 šumskih parcela")

 Uneto 6 šumskih parcela


In [6]:
# Unos podataka u tabelu vatrogasne jedinice

vatrogasne = [
    ("VJ Pirot", "Nikole Pašića 12, Pirot", 6, 24, "24h", 1, "POINT(22.5853 43.1525)"),
    ("VJ Babušnica", "Centar 5, Babušnica", 2, 8, "24h", 2, "POINT(22.4167 43.1167)"),
    ("VJ Dimitrovgrad", "Glavna 3, Dimitrovgrad", 3, 10, "24h", 3, "POINT(22.7667 43.0167)"),
    ("VJ Knjaževac", "Stevana Sinđelića 1, Knjaževac", 4, 14, "24h", 4, "POINT(22.2667 43.5667)"),
    ("VJ Zaječar", "Trg oslobođenja 2, Zaječar", 7, 22, "24h", 5, "POINT(22.2858 43.9038)"),
]

for naziv, adresa, vozila, osoblje, radno_vreme, opstina_id, geom in vatrogasne:
    cur.execute("""
        INSERT INTO vatrogasne_jedinice 
        (naziv, adresa, broj_vozila, broj_osoblja, radno_vreme, opstina_id, geom)
        VALUES (%s, %s, %s, %s, %s, %s, ST_GeomFromText(%s, 4326))
    """, (naziv, adresa, vozila, osoblje, radno_vreme, opstina_id, geom))

conn.commit()
print(" Uneto 5 vatrogasnih jedinica")

 Uneto 5 vatrogasnih jedinica


In [7]:
# Unos podataka u tabelu meteorološke stanice

stanice = [
    ("Meteo Pirot", 380, "automatska", 1, "POINT(22.60 43.16)"),
    ("Meteo Babušnica", 450, "automatska", 2, "POINT(22.43 43.12)"),
    ("Meteo Dimitrovgrad", 420, "automatska", 3, "POINT(22.78 43.02)"),
    ("Meteo Knjaževac", 510, "automatska", 4, "POINT(22.28 43.58)"),
    ("Meteo Stara planina vrh", 1500, "ručna", 4, "POINT(22.23 43.40)"),
]

for naziv, visina, tip, opstina_id, geom in stanice:
    cur.execute("""
        INSERT INTO meteo_stanice (naziv, visina_m, tip_stanice, opstina_id, geom)
        VALUES (%s, %s, %s, %s, ST_GeomFromText(%s, 4326))
    """, (naziv, visina, tip, opstina_id, geom))

conn.commit()
print(" Uneto 5 meteo stanica")

 Uneto 5 meteo stanica


In [8]:
# Unos podataka u tabelu meteorološka merenja

import random
from datetime import datetime, timedelta

# Generišemo više meteo podataka za ceo period
start_date = datetime(2024, 6, 1)
end_date = datetime(2024, 10, 31)
dates = []
current = start_date
while current <= end_date:
    dates.append(current)
    current += timedelta(days=1)

meteo_podaci = []

for datum in dates:
    for stanica_id in range(1, 6):  # 5 stanica
        temperatura = round(random.uniform(15, 40), 1)
        vlaznost = random.randint(15, 85)
        vetar = round(random.uniform(0, 45), 1)
        padavine = round(random.uniform(0, 20), 1)
        
        meteo_podaci.append((stanica_id, datum, temperatura, vlaznost, vetar, padavine))

# Unos u bazu
for stanica_id, datum, temp, vlaznost, vetar, padavine in meteo_podaci:
    cur.execute("""
        INSERT INTO meteo_merenja 
        (stanica_id, datum, temperatura, vlaznost_vazduha, brzina_vetra, padavine_mm)
        VALUES (%s, %s, %s, %s, %s, %s)
    """, (stanica_id, datum, temp, vlaznost, vetar, padavine))

conn.commit()
print(f" Uneto {len(meteo_podaci)} meteo merenja!")

 Uneto 765 meteo merenja!


In [9]:
# Unos podataka u tabelu šumski požari

pozari = [
    ("2024-07-10", "2024-07-12", 1, "ljudski faktor (loza)", "visok", 45.30, 18),
    ("2024-07-22", "2024-07-22", 2, "udar groma", "nizak", 3.10, 6),
    ("2024-08-02", "2024-08-05", 3, "ljudski faktor (spaljivanje korova)", "srednji", 22.70, 12),
    ("2024-08-03", "2024-08-06", 5, "nepoznat", "visok", 60.00, 25),
    ("2024-08-15", "2024-08-15", 6, "udar groma", "nizak", 5.50, 8),
]

for datum_poc, datum_gas, parcela_id, uzrok, intenzitet, povrsina, vatrogasci in pozari:
    cur.execute("""
        INSERT INTO pozari 
        (datum_pocetka, datum_gasenja, parcela_id, uzrok, intenzitet, 
         povrsina_zahvacena_ha, broj_angazovanih_vatrogasaca)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
    """, (datum_poc, datum_gas, parcela_id, uzrok, intenzitet, povrsina, vatrogasci))

conn.commit()
print(" Uneto 5 požara")

 Uneto 5 požara


In [10]:
# Unos podataka u tabelu opožarena područja

opozarena = [
    (1, "2024-07-13", "NBR_threshold", 44.10, 0.412,
     "POLYGON((22.555 43.105, 22.59 43.105, 22.59 43.14, 22.555 43.14, 22.555 43.105))"),
    (2, "2024-07-23", "NBR_threshold", 3.00, 0.355,
     "POLYGON((22.36 43.06, 22.38 43.06, 22.38 43.08, 22.36 43.08, 22.36 43.06))"),
    (3, "2024-08-06", "NBR_threshold", 21.90, 0.480,
     "POLYGON((22.71 42.98, 22.74 42.98, 22.74 43.00, 22.71 43.00, 22.71 42.98))"),
    (4, "2024-08-07", "NBR_threshold", 58.70, 0.520,
     "POLYGON((22.21 43.51, 22.26 43.51, 22.26 43.56, 22.21 43.56, 22.21 43.51))"),
    (5, "2024-08-16", "NBR_threshold", 5.20, 0.330,
     "POLYGON((22.21 43.86, 22.25 43.86, 22.25 43.90, 22.21 43.90, 22.21 43.86))"),
]

for pozar_id, datum, metoda, povrsina, indeks, geom in opozarena:
    cur.execute("""
        INSERT INTO opozarena_podrucja 
        (pozar_id, datum_detekcije, metoda_detekcije, povrsina_ha, indeks_dnbr, geom)
        VALUES (%s, %s, %s, %s, %s, ST_GeomFromText(%s, 4326))
    """, (pozar_id, datum, metoda, povrsina, indeks, geom))

conn.commit()
print(" Uneto 5 opožarenih područja")

 Uneto 5 opožarenih područja


In [11]:
# Provera unosa podataka u sve tabele

tabele = ['opstine', 'sumske_parcele', 'vatrogasne_jedinice', 
          'meteo_stanice', 'meteo_merenja', 'pozari', 'opozarena_podrucja']

for tabela in tabele:
    cur.execute(f"SELECT COUNT(*) FROM {tabela}")
    broj = cur.fetchone()[0]
    print(f" {tabela}: {broj} redova")

print("\n Svi podaci uspešno uneti!")

 opstine: 5 redova
 sumske_parcele: 6 redova
 vatrogasne_jedinice: 5 redova
 meteo_stanice: 5 redova
 meteo_merenja: 765 redova
 pozari: 5 redova
 opozarena_podrucja: 5 redova

 Svi podaci uspešno uneti!


In [15]:
#zatvaranje konekcije

cur.close()
conn.close()
print("Konekcija zatvorena")

Konekcija zatvorena


In [16]:
#ponovo povezivanje na bazu

conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()
print("Ponovo povezani na bazu!")

Ponovo povezani na bazu!


In [17]:
#crud operacije za tabele

def citaj_sve_pozare():
    """Čita sve požare sa detaljima"""
    query = """
        SELECT p.id, p.datum_pocetka, p.datum_gasenja, 
               s.naziv as parcela, p.uzrok, p.intenzitet,
               p.povrsina_zahvacena_ha, p.broj_angazovanih_vatrogasaca
        FROM pozari p
        LEFT JOIN sumske_parcele s ON p.parcela_id = s.id
        ORDER BY p.datum_pocetka DESC
    """
    return pd.read_sql(query, conn)

# Prikaz svih požara
df_pozari = citaj_sve_pozare()
print("Svi požari:")
display(df_pozari)

Svi požari:


,id,datum_pocetka,datum_gasenja,parcela,uzrok,intenzitet,povrsina_zahvacena_ha,broj_angazovanih_vatrogasaca
0,5,2024-08-15,2024-08-15,Parcela Crni vrh,udar groma,nizak,5.5,8
1,4,2024-08-03,2024-08-06,Parcela Stara planina sever,nepoznat,visok,60.0,25
2,3,2024-08-02,2024-08-05,Parcela Petrlaš,ljudski faktor (spaljivanje korova),srednji,22.7,12
3,2,2024-07-22,2024-07-22,Parcela Rsovci,udar groma,nizak,3.1,6
4,1,2024-07-10,2024-07-12,Parcela Vlasina,ljudski faktor (loza),visok,45.3,18


In [18]:
# Crud operacija create

def dodaj_pozar(datum_pocetka, datum_gasenja, parcela_id, uzrok, intenzitet, povrsina, vatrogasci):
    """Dodaje novi požar u bazu"""
    try:
        cur.execute("""
            INSERT INTO pozari 
            (datum_pocetka, datum_gasenja, parcela_id, uzrok, intenzitet, 
             povrsina_zahvacena_ha, broj_angazovanih_vatrogasaca)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
            RETURNING id
        """, (datum_pocetka, datum_gasenja, parcela_id, uzrok, intenzitet, povrsina, vatrogasci))
        conn.commit()
        novi_id = cur.fetchone()[0]
        print(f"✅ Novi požar dodat sa ID: {novi_id}")
        return novi_id
    except Exception as e:
        print(f" Greška: {e}")
        return None

# Primer - dodajemo novi požar
dodaj_pozar(
    datum_pocetka="2024-09-01",
    datum_gasenja="2024-09-02",
    parcela_id=1,
    uzrok="ljudski faktor (kampovanje)",
    intenzitet="srednji",
    povrsina=15.20,
    vatrogasci=10
)

# Proveravamo da li je dodat
df_pozari = citaj_sve_pozare()
print("\nPožari nakon dodavanja:")
display(df_pozari)

✅ Novi požar dodat sa ID: 6

Požari nakon dodavanja:


,id,datum_pocetka,datum_gasenja,parcela,uzrok,intenzitet,povrsina_zahvacena_ha,broj_angazovanih_vatrogasaca
0,6,2024-09-01,2024-09-02,Parcela Vlasina,ljudski faktor (kampovanje),srednji,15.2,10
1,5,2024-08-15,2024-08-15,Parcela Crni vrh,udar groma,nizak,5.5,8
2,4,2024-08-03,2024-08-06,Parcela Stara planina sever,nepoznat,visok,60.0,25
3,3,2024-08-02,2024-08-05,Parcela Petrlaš,ljudski faktor (spaljivanje korova),srednji,22.7,12
4,2,2024-07-22,2024-07-22,Parcela Rsovci,udar groma,nizak,3.1,6
5,1,2024-07-10,2024-07-12,Parcela Vlasina,ljudski faktor (loza),visok,45.3,18


In [19]:
# Crud operacija update

def azuriraj_pozar(pozar_id, nova_povrsina, novi_intenzitet):
    """Ažurira površinu i intenzitet požara"""
    try:
        cur.execute("""
            UPDATE pozari 
            SET povrsina_zahvacena_ha = %s, intenzitet = %s
            WHERE id = %s
        """, (nova_povrsina, novi_intenzitet, pozar_id))
        conn.commit()
        print(f" Požar ID {pozar_id} ažuriran!")
        return True
    except Exception as e:
        print(f" Greška: {e}")
        return False

# Ažuriramo poslednji dodati požar (uzmi njegov ID)
cur.execute("SELECT MAX(id) FROM pozari")
zadnji_id = cur.fetchone()[0]

if zadnji_id:
    azuriraj_pozar(zadnji_id, 18.50, "visok")
    
# Proveravamo promene
df_pozari = citaj_sve_pozare()
print("\n Požari nakon ažuriranja:")
display(df_pozari.head(3))

 Požar ID 6 ažuriran!

 Požari nakon ažuriranja:


,id,datum_pocetka,datum_gasenja,parcela,uzrok,intenzitet,povrsina_zahvacena_ha,broj_angazovanih_vatrogasaca
0,6,2024-09-01,2024-09-02,Parcela Vlasina,ljudski faktor (kampovanje),visok,18.5,10
1,5,2024-08-15,2024-08-15,Parcela Crni vrh,udar groma,nizak,5.5,8
2,4,2024-08-03,2024-08-06,Parcela Stara planina sever,nepoznat,visok,60.0,25


In [20]:
# CRUD OPERACIJE - DELETE

def obrisi_pozar(pozar_id):
    """Briše požar iz baze"""
    try:
        cur.execute("DELETE FROM pozari WHERE id = %s", (pozar_id,))
        conn.commit()
        print(f" Požar ID {pozar_id} obrisan!")
        return True
    except Exception as e:
        print(f" Greška: {e}")
        return False

# Brišemo poslednji dodati požar
if zadnji_id:
    obrisi_pozar(zadnji_id)

# Proveravamo
df_pozari = citaj_sve_pozare()
print("\n Požari nakon brisanja:")
display(df_pozari)

 Požar ID 6 obrisan!

 Požari nakon brisanja:


,id,datum_pocetka,datum_gasenja,parcela,uzrok,intenzitet,povrsina_zahvacena_ha,broj_angazovanih_vatrogasaca
0,5,2024-08-15,2024-08-15,Parcela Crni vrh,udar groma,nizak,5.5,8
1,4,2024-08-03,2024-08-06,Parcela Stara planina sever,nepoznat,visok,60.0,25
2,3,2024-08-02,2024-08-05,Parcela Petrlaš,ljudski faktor (spaljivanje korova),srednji,22.7,12
3,2,2024-07-22,2024-07-22,Parcela Rsovci,udar groma,nizak,3.1,6
4,1,2024-07-10,2024-07-12,Parcela Vlasina,ljudski faktor (loza),visok,45.3,18


In [21]:
# JOIN UPITI - 5 PRIMERA
print("=" * 50)
print("1. POŽARI SA PODACIMA O ŠUMSKIM PARCELAMA")
print("=" * 50)

query1 = """
    SELECT p.id, p.datum_pocetka, p.intenzitet, 
           s.naziv as parcela, s.tip_sume, s.povrsina_ha
    FROM pozari p
    JOIN sumske_parcele s ON p.parcela_id = s.id
    ORDER BY p.datum_pocetka DESC
"""
print(pd.read_sql(query1, conn))

print("\n" + "=" * 50)
print("2. POŽARI SA BROJEM VATROGASACA I OPŠTINOM")
print("=" * 50)

query2 = """
    SELECT p.id, p.datum_pocetka, p.broj_angazovanih_vatrogasaca,
           o.naziv as opstina, o.region
    FROM pozari p
    JOIN sumske_parcele s ON p.parcela_id = s.id
    JOIN opstine o ON s.opstina_id = o.id
    ORDER BY p.broj_angazovanih_vatrogasaca DESC
"""
print(pd.read_sql(query2, conn))

print("\n" + "=" * 50)
print("3. METEOROLOŠKA MERENJA ZA OPŠTINE SA POŽARIMA")
print("=" * 50)

query3 = """
    SELECT m.datum, m.temperatura, m.vlaznost_vazduha, 
           o.naziv as opstina,
           COUNT(p.id) as broj_pozara
    FROM meteo_merenja m
    JOIN meteo_stanice ms ON m.stanica_id = ms.id
    JOIN opstine o ON ms.opstina_id = o.id
    LEFT JOIN pozari p ON p.datum_pocetka = m.datum
    GROUP BY m.datum, m.temperatura, m.vlaznost_vazduha, o.naziv
    ORDER BY m.datum DESC
"""
print(pd.read_sql(query3, conn))

print("\n" + "=" * 50)
print("4. UKUPNA POVRŠINA POŽARA PO OPŠTINAMA")
print("=" * 50)

query4 = """
    SELECT o.naziv as opstina, 
           COUNT(p.id) as broj_pozara,
           SUM(p.povrsina_zahvacena_ha) as ukupna_povrsina_ha,
           AVG(p.broj_angazovanih_vatrogasaca) as prosecno_vatrogasaca
    FROM opstine o
    LEFT JOIN sumske_parcele s ON o.id = s.opstina_id
    LEFT JOIN pozari p ON s.id = p.parcela_id
    GROUP BY o.naziv
    ORDER BY ukupna_povrsina_ha DESC
"""
print(pd.read_sql(query4, conn))

print("\n" + "=" * 50)
print("5. OPOŽARENA PODRUČJA SA DETALJIMA POŽARA")
print("=" * 50)

query5 = """
    SELECT op.datum_detekcije, op.povrsina_ha, op.indeks_dnbr,
           p.datum_pocetka, p.intenzitet, p.uzrok,
           s.naziv as parcela
    FROM opozarena_podrucja op
    JOIN pozari p ON op.pozar_id = p.id
    JOIN sumske_parcele s ON p.parcela_id = s.id
    ORDER BY op.povrsina_ha DESC
"""
print(pd.read_sql(query5, conn))

1. POŽARI SA PODACIMA O ŠUMSKIM PARCELAMA
   id datum_pocetka intenzitet                      parcela       tip_sume  \
0   5    2024-08-15      nizak             Parcela Crni vrh    borova šuma   
1   4    2024-08-03      visok  Parcela Stara planina sever  mesovita šuma   
2   3    2024-08-02    srednji              Parcela Petrlaš    borova šuma   
3   2    2024-07-22      nizak               Parcela Rsovci  mesovita šuma   
4   1    2024-07-10      visok              Parcela Vlasina    bukova šuma   

   povrsina_ha  
0        198.4  
1        410.0  
2        150.0  
3        210.2  
4        340.5  

2. POŽARI SA BROJEM VATROGASACA I OPŠTINOM
   id datum_pocetka  broj_angazovanih_vatrogasaca    opstina          region
0   4    2024-08-03                            25  Knjaževac   Stara planina
1   1    2024-07-10                            18      Pirot   Stara planina
2   3    2024-08-02                            12  Babušnica   Stara planina
3   5    2024-08-15                

In [22]:
# DODATNIH 5 JOIN UPITA SA WHERE FILTRIRANJEM

print("=" * 60)
print("UPIT 1: Požari visokog intenziteta sa površinom preko 20ha")
print("=" * 60)

query1 = """
    SELECT p.id, p.datum_pocetka, p.intenzitet, 
           p.povrsina_zahvacena_ha,
           s.naziv as parcela, s.tip_sume,
           o.naziv as opstina
    FROM pozari p
    JOIN sumske_parcele s ON p.parcela_id = s.id
    JOIN opstine o ON s.opstina_id = o.id
    WHERE p.intenzitet = 'visok' 
      AND p.povrsina_zahvacena_ha > 20
    ORDER BY p.povrsina_zahvacena_ha DESC
"""
print(pd.read_sql(query1, conn))


print("\n" + "=" * 60)
print("UPIT 2: Požari koji su trajali više od 2 dana")
print("=" * 60)

query2 = """
    SELECT p.id, p.datum_pocetka, p.datum_gasenja,
           (p.datum_gasenja - p.datum_pocetka) as trajanje_dana,
           p.povrsina_zahvacena_ha,
           s.naziv as parcela
    FROM pozari p
    JOIN sumske_parcele s ON p.parcela_id = s.id
    WHERE p.datum_gasenja IS NOT NULL 
      AND (p.datum_gasenja - p.datum_pocetka) > 2
    ORDER BY trajanje_dana DESC
"""
print(pd.read_sql(query2, conn))


print("\n" + "=" * 60)
print("UPIT 3: Meteorološka merenja na dan požara sa temperaturom preko 33°C")
print("=" * 60)

query3 = """
    SELECT p.id as pozar_id, p.datum_pocetka, 
           p.intenzitet, p.povrsina_zahvacena_ha,
           m.temperatura, m.vlaznost_vazduha, m.brzina_vetra,
           ms.naziv as meteo_stanica
    FROM pozari p
    JOIN meteo_merenja m ON p.datum_pocetka = m.datum
    JOIN meteo_stanice ms ON m.stanica_id = ms.id
    WHERE m.temperatura > 33
    ORDER BY m.temperatura DESC
"""
print(pd.read_sql(query3, conn))


print("\n" + "=" * 60)
print("UPIT 4: Požari u opštini Pirot sa više od 10 vatrogasaca")
print("=" * 60)

query4 = """
    SELECT p.id, p.datum_pocetka, p.intenzitet,
           p.broj_angazovanih_vatrogasaca,
           p.povrsina_zahvacena_ha,
           o.naziv as opstina
    FROM pozari p
    JOIN sumske_parcele s ON p.parcela_id = s.id
    JOIN opstine o ON s.opstina_id = o.id
    WHERE o.naziv = 'Pirot' 
      AND p.broj_angazovanih_vatrogasaca > 10
    ORDER BY p.broj_angazovanih_vatrogasaca DESC
"""
print(pd.read_sql(query4, conn))


print("\n" + "=" * 60)
print("UPIT 5: Opožarena područja sa indeksom dNBR preko 0.40")
print("=" * 60)

query5 = """
    SELECT op.id, op.datum_detekcije, 
           op.povrsina_ha, op.indeks_dnbr,
           p.datum_pocetka, p.intenzitet,
           s.naziv as parcela
    FROM opozarena_podrucja op
    JOIN pozari p ON op.pozar_id = p.id
    JOIN sumske_parcele s ON p.parcela_id = s.id
    WHERE op.indeks_dnbr > 0.40
    ORDER BY op.indeks_dnbr DESC
"""
print(pd.read_sql(query5, conn))

UPIT 1: Požari visokog intenziteta sa površinom preko 20ha
   id datum_pocetka intenzitet  povrsina_zahvacena_ha  \
0   4    2024-08-03      visok                   60.0   
1   1    2024-07-10      visok                   45.3   

                       parcela       tip_sume    opstina  
0  Parcela Stara planina sever  mesovita šuma  Knjaževac  
1              Parcela Vlasina    bukova šuma      Pirot  

UPIT 2: Požari koji su trajali više od 2 dana
   id datum_pocetka datum_gasenja  trajanje_dana  povrsina_zahvacena_ha  \
0   3    2024-08-02    2024-08-05              3                   22.7   
1   4    2024-08-03    2024-08-06              3                   60.0   

                       parcela  
0              Parcela Petrlaš  
1  Parcela Stara planina sever  

UPIT 3: Meteorološka merenja na dan požara sa temperaturom preko 33°C
   pozar_id datum_pocetka intenzitet  povrsina_zahvacena_ha  temperatura  \
0         2    2024-07-22      nizak                    3.1         39.3 

In [23]:
# JOIN + GROUP BY + HAVING (filtriranje grupa)

print("=" * 60)
print("UPIT 6: Opštine sa ukupnom površinom požara preko 50ha")
print("=" * 60)

query6 = """
    SELECT o.naziv as opstina, 
           COUNT(p.id) as broj_pozara,
           SUM(p.povrsina_zahvacena_ha) as ukupna_povrsina_ha,
           AVG(p.broj_angazovanih_vatrogasaca) as prosecno_vatrogasaca
    FROM opstine o
    JOIN sumske_parcele s ON o.id = s.opstina_id
    LEFT JOIN pozari p ON s.id = p.parcela_id
    GROUP BY o.naziv
    HAVING SUM(p.povrsina_zahvacena_ha) > 50
    ORDER BY ukupna_povrsina_ha DESC
"""
print(pd.read_sql(query6, conn))


print("\n" + "=" * 60)
print("UPIT 7: Šumske parcele sa više od 1 požara")
print("=" * 60)

query7 = """
    SELECT s.naziv as parcela, s.tip_sume,
           COUNT(p.id) as broj_pozara,
           SUM(p.povrsina_zahvacena_ha) as ukupna_povrsina
    FROM sumske_parcele s
    LEFT JOIN pozari p ON s.id = p.parcela_id
    GROUP BY s.naziv, s.tip_sume
    HAVING COUNT(p.id) >= 1
    ORDER BY broj_pozara DESC
"""
print(pd.read_sql(query7, conn))

UPIT 6: Opštine sa ukupnom površinom požara preko 50ha
     opstina  broj_pozara  ukupna_povrsina_ha  prosecno_vatrogasaca
0  Knjaževac            1                60.0                  25.0

UPIT 7: Šumske parcele sa više od 1 požara
                       parcela       tip_sume  broj_pozara  ukupna_povrsina
0  Parcela Stara planina sever  mesovita šuma            1             60.0
1               Parcela Rsovci  mesovita šuma            1              3.1
2             Parcela Crni vrh    borova šuma            1              5.5
3              Parcela Petrlaš    borova šuma            1             22.7
4              Parcela Vlasina    bukova šuma            1             45.3


In [24]:
# ČUVANJE PODATAKA U DATAFRAME

# Učitavamo sve tabele u DataFrame-ove
df_opstine = pd.read_sql("SELECT * FROM opstine", conn)
df_parcele = pd.read_sql("SELECT * FROM sumske_parcele", conn)
df_vatrogasne = pd.read_sql("SELECT * FROM vatrogasne_jedinice", conn)
df_stanice = pd.read_sql("SELECT * FROM meteo_stanice", conn)
df_merenja = pd.read_sql("SELECT * FROM meteo_merenja", conn)
df_pozari = pd.read_sql("SELECT * FROM pozari", conn)
df_opozarena = pd.read_sql("SELECT * FROM opozarena_podrucja", conn)

print(" Svi podaci učitani u DataFrame-ove:")
print(f" opstine: {len(df_opstine)} redova")
print(f" sumske_parcele: {len(df_parcele)} redova")
print(f" vatrogasne_jedinice: {len(df_vatrogasne)} redova")
print(f" meteo_stanice: {len(df_stanice)} redova")
print(f" meteo_merenja: {len(df_merenja)} redova")
print(f" pozari: {len(df_pozari)} redova")
print(f" opozarena_podrucja: {len(df_opozarena)} redova")

 Svi podaci učitani u DataFrame-ove:
 opstine: 5 redova
 sumske_parcele: 6 redova
 vatrogasne_jedinice: 5 redova
 meteo_stanice: 5 redova
 meteo_merenja: 765 redova
 pozari: 5 redova
 opozarena_podrucja: 5 redova
